# Single Reciter E2E Pipeline (Colab GPU)

**Config:** `--device cuda --cuda-batch-size 8 --intra-surah-split --max-workers 1`
**Output:** Opus audio (ffmpeg loudnorm) + JSON word-timings per surah → Google Drive

Select reciter from `verified_all_curl_cffi.json` (226 verified with 114+ surahs).

In [ ]:
# 1. Install quran-forced-align with CUDA
!pip install -q git+https://github.com/HsnSaboor/quran-forced-align.git --extra cuda 2>/dev/null || \
!pip install -q -e . --extra cuda
import os, json, urllib.request
os.makedirs('model', exist_ok=True)
if not os.path.exists('model/zipformer_p_arabic_v2.int8.onnx'):
    print('⚠ Model not found. Download from HF: Muno459/zipformer_p-arabic-v2')

In [ ]:
# 2. Mount Drive + setup dirs
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/quran_forced_align'
AUDIO_DIR  = f'{DRIVE_ROOT}/audio_input'
OUTPUT_DIR = f'{DRIVE_ROOT}/output'
OPUS_DIR   = f'{OUTPUT_DIR}/opus'
JSON_DIR   = f'{OUTPUT_DIR}/json'
!mkdir -p {AUDIO_DIR} {OUTPUT_DIR} {OPUS_DIR} {JSON_DIR}

## 3. Select Reciter (from verified 226)
Edit `RECITER` below. Must exist in `verified_all_curl_cffi.json`.

In [ ]:
# 3a. Reciter config — pick one from verified_all_curl_cffi.json
RECITER = {
    'name': 'mishary-rashid-alafasy',   # slug
    'reciter_id': 1,                   # numeric ID from assabile
    'collection_id': 1,                # best collection (most plays, 114 unique surahs)
    'enhance': True                    # ffmpeg loudnorm + opus encode
}

# Verify reciter in verified list
with open('verified_all_curl_cffi.json') as f:
    v = json.load(f)
    match = next((r for r in v['verified'] if r['slug'] == RECITER['name']), None)
    if match:
        print(f"✓ Verified: {match['slug']} coll={match['collection_id']} unique={match['unique_surahs']} total={match['total_recs']}")
        if match['collection_id'] != RECITER['collection_id']:
            print(f"  ⚠ collection_id mismatch! File has {match['collection_id']}, config has {RECITER['collection_id']}")
    else:
        print(f"✗ {RECITER['name']} not in verified list!")

RECITER_SLUG = RECITER['name']
RECITER_ID = RECITER['reciter_id']
COLLECTION_ID = RECITER['collection_id']
AUDIO_OUT = f'{AUDIO_DIR}/{RECITER_SLUG}'
JSON_OUT = f'{JSON_DIR}/{RECITER_SLUG}'
OPUS_OUT = f'{OPUS_DIR}/{RECITER_SLUG}'
!mkdir -p {AUDIO_OUT} {JSON_OUT} {OPUS_OUT}

In [ ]:
# 4. Download all 114 surah audio files via ajax/getrcita-link
import asyncio
from curl_cffi.requests import AsyncSession

async def download_reciter(reciter_id, collection_id, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    async with AsyncSession(timeout=10, impersonate='chrome120') as session:
        # First get loadplayer to discover surah href IDs
        lp_url = f'https://www.assabile.com/ajax/loadplayer-{reciter_id}-{collection_id}'
        resp = await session.get(lp_url)
        if resp.status_code != 200:
            raise Exception(f'loadplayer failed: {resp.status_code}')
        data = resp.json()
        recs = data.get('Recitation', [])
        print(f'Found {len(recs)} recitations in loadplayer')
        
        # Download each surah
        sem = asyncio.Semaphore(10)
        async def download_one(r):
            surah = r.get('sura_id')
            href = r.get('href', '').lstrip('#')
            if not surah or not href:
                return False
            async with sem:
                try:
                    dl_url = f'https://www.assabile.com/ajax/getrcita-link-{href}'
                    r2 = await session.get(f'https://www.assabile.com/ajax/getrcita-link-{href}', timeout=8)
                    if r2.status_code != 200:
                        return False
                    mp3_url = r2.text.strip()
                    # Download MP3
                    r3 = await session.get(mp3_url, timeout=30)
                    if r3.status_code == 200:
                        fname = f'{out_dir}/{int(surah):03d}.mp3'
                        with open(fname, 'wb') as f:
                            f.write(r3.content)
                        return True
                except Exception as e:
                    print(f'  Surah {surah} failed: {e}')
            return False
        
        tasks = [download_one(r) for r in recs]
        results = await asyncio.gather(*tasks)
        return sum(results)

count = asyncio.run(download_reciter(RECITER_ID, COLLECTION_ID, AUDIO_OUT))
print(f'Downloaded {count}/114 surahs to {AUDIO_OUT}')

In [ ]:
# 5. Verify downloaded files
import glob
files = sorted(glob.glob(f'{AUDIO_OUT}/*.mp3'))
print(f'Downloaded {len(files)} MP3 files')
if len(files) < 114:
    print('⚠ Less than 114 files — alignment may be incomplete')
    missing = set(range(1,115)) - {int(os.path.basename(f).split('.')[0]) for f in files}
    print(f'Missing surahs: {sorted(missing)}')
else:
    print('✓ All 114 surahs present')

In [ ]:
# 6. Convert MP3 → Opus with loudnorm enhancement
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
import subprocess, os

def mp3_to_opus(mp3_path, opus_path):
    os.makedirs(os.path.dirname(opus_path) or '.', exist_ok=True)
    cmd = [
        'ffmpeg', '-y', '-i', mp3_path,
        '-af', 'loudnorm=I=-16:TP=-1.5:LRA=11,acompressor=threshold=-25dB:ratio=3:attack=5:release=50',
        '-c:a', 'libopus', '-b:a', '96k', '-vbr', 'on',
        '-application', 'audio', '-frame_duration', '60',
        opus_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return os.path.exists(opus_path)

print('Converting to Opus...')
opus_count = 0
for f in sorted(glob.glob(f'{AUDIO_OUT}/*.mp3')):
    surah = os.path.basename(f).split('.')[0]
    opus_path = f'{OPUS_OUT}/{surah}.opus'
    if mp3_to_opus(f, opus_path):
        opus_count += 1
print(f'Converted {opus_count} files to Opus in {OPUS_OUT}')

In [ ]:
# 7. Run quran-forced-align-batch (GPU optimized)
import time, subprocess

BATCH_SIZE = 8
MAX_WORKERS = 1
DEVICE = 'cuda'

# Find available surahs
available = sorted([int(f.split('.')[0]) for f in os.listdir(AUDIO_OUT) if f.endswith('.mp3')])
if not available:
    raise Exception('No MP3 files found')

print(f'Aligning surahs {min(available)}-{max(available)} ({len(available)} surahs)')
print(f'Config: --device {DEVICE} --cuda-batch-size {BATCH_SIZE} --intra-surah-split --max-workers {MAX_WORKERS}')

cmd = [
    'python', '-m', 'quran_forced_align.batch_cli',
    '--surahs', f'{min(available)}-{max(available)}',
    '--audio-dir', AUDIO_OUT,
    '--out-dir', JSON_OUT,
    '--device', DEVICE,
    '--cuda-batch-size', str(BATCH_SIZE),
    '--intra-surah-split',
    '--max-workers', str(MAX_WORKERS),
    '--model', 'model/zipformer_p_arabic_v2.int8.onnx',
    '--tokens', 'model/tokens.txt',
]

t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - t0

print(result.stdout[-3000:] if result.stdout else '')
if result.stderr:
    print('STDERR:', result.stderr[-1000:])
print(f'Alignment done: {elapsed:.1f}s, return_code={result.returncode}')

In [ ]:
# 8. Verify JSON outputs + copy to Drive
import shutil

json_files = sorted(glob.glob(f'{JSON_OUT}/*.json'))
opus_files = sorted(glob.glob(f'{OPUS_OUT}/*.opus'))
print(f'Generated {len(json_files)} JSON, {len(opus_files)} Opus')

# Copy to Drive (structured by reciter)
DRIVE_REC_OPUS = f'{DRIVE_ROOT}/opus/{RECITER_SLUG}'
DRIVE_REC_JSON = f'{DRIVE_ROOT}/json/{RECITER_SLUG}'
os.makedirs(DRIVE_REC_OPUS, exist_ok=True)
os.makedirs(DRIVE_REC_JSON, exist_ok=True)

for f in opus_files:
    shutil.copy2(f, DRIVE_REC_OPUS)
for f in json_files:
    shutil.copy2(f, DRIVE_REC_JSON)

print(f'✓ Copied to Drive:')
print(f'  Opus: {DRIVE_REC_OPUS} ({len(opus_files)} files)')
print(f'  JSON: {DRIVE_REC_JSON} ({len(json_files)} files)')

In [ ]:
# 9. Summary report
print('=' * 60)
print(f'RECITER: {RECITER_SLUG}')
print(f'Surahs processed: {len(available)}')
print(f'Opus files: {len(opus_files)} → {DRIVE_REC_OPUS}')
print(f'JSON files: {len(json_files)} → {DRIVE_REC_JSON}')
print(f'GPU config: batch_size={BATCH_SIZE}, intra_surah_split=True, max_workers={MAX_WORKERS}')
print('=' * 60)

# Quick sanity check on first JSON
if json_files:
    import json
    with open(json_files[0]) as f:
        data = json.load(f)
    print(f'Sample JSON (surah {data[0]["sura"]}): {len(data)} words, {sum(1 for w in data if w.get("is_repeat"))} repeats')